# ir_calendar_init_tables — 一次性建表（bronze 3 張 + silver 7 張）

- 用途：為法說行事曆（ir_calendar）批次消費 job 建立 Delta table。爬蟲每批產出除了落 Volume、轉送系統 API，也在 Databricks 留一份：bronze 原樣保存、silver 供查詢。
- 執行次數：一次。可安全重跑：全部 `CREATE TABLE IF NOT EXISTS`，不動既有資料；只有 `recreate = YES_DROP_ALL` 才會 DROP。上線後才加的欄位由 [c21] `ALTER TABLE ADD COLUMNS` 補到既有表（已有就略過）。
- 輸入 table：無。
- 輸出 table：`{catalog}.{schema}.b_{domain}_*` 3 張、`s_{domain}_*` 6 張，清單見 [c02]。前 6 張給 `consume_batches`，後 3 張（[c16]～[c18]）給 `parse_documents`。`domain` 預設 `ir_calendar`。
- 參數（widgets）：`catalog`、`schema`、`domain`、`recreate`。預設值對照 `config/project.yml`。
- 排程：無（手動執行一次）。
- 負責人 / 更新日期：（填）/ 2026-09-21
- 設計說明：`docs/20260921_ir_calendar_lakehouse_design.md`；分層通則：`docs/conventions.md` 2.1。

## 分層

| 層 | 表 | 性質 | 內容 |
|---|---|---|---|
| bronze | `b_ir_calendar_record` | **append-only**，schema 固定 | 爬蟲批次裡每個檔案一列：`record_type` + `payload`（JSON 全文，STRING）+ 批次血緣。所有 record_type 共用這一張 |
| bronze | `b_ir_calendar_batch_log` | 以 `batch_id` MERGE | 每批的 manifest 摘要、處理結果、API 回應（job 遙測，不是來源資料，所以允許覆蓋） |
| silver | `s_ir_calendar_conference` | MERGE，鍵 `(company_key, period, revision)` | 法說場次，每個 revision 一列 |
| silver | `s_ir_calendar_summary` | MERGE，鍵 `(company_key, period)` | 法說彙整（系統端無對應表） |
| silver | `s_ir_calendar_document_file` | MERGE，鍵 `volume_path` | 實體檔案索引（Volume 上每個 PDF / HTM） |
| silver | `s_ir_calendar_company` | MERGE，鍵 `company_key` | 公司主表快照 |
| silver | `s_ir_calendar_financials` | MERGE，鍵 `(company_key, period)` | 財報數據（`ir_financials`）：年度 `yyyyFY` + 季別 `yyyyQn` 一期一列，`period_type` 區分；台股 TWD 仟元 / 美股 USD 元（[c19]，2026-10 新增） |
| bronze | `b_ir_calendar_document_parse` | 以 `(volume_path, sha256)` 刪後 append | Volume 財報檔案的 `ai_parse_document` 原始結果：`payload` = 輸出 JSON 全文 + status / 頁數 / 元素數。`category` / `company_key` 是欄位，新分類自動進來 |
| silver | `s_ir_calendar_document_text` | MERGE，鍵 `(volume_path, sha256)` | 一檔一列，全文 markdown |
| silver | `s_ir_calendar_document_element` | 以文件為單位刪後 append，鍵 `(volume_path, sha256, element_id)` | 一元素一列（段落 / 表格 / 圖表描述），給切塊與抽取 |

silver 全部可以從 bronze 重算（`consume_batches` / `parse_documents` 各自的 `rebuild_silver = true`），不必碰 Volume 歸檔或重新呼叫 `ai_parse_document`。

## 維護者須知

| 事項 | 說明 |
|---|---|
| 表命名 | `<層級>_<領域>_<短名>`：`b_` 原始、`s_` 銀質、`g_` 金質、`app_` 其他系統來的。這裡的層級前綴固定，只有 `domain` 可改。 |
| 爬蟲加欄位 | bronze **不用改**（`payload` 整包收）。silver 想用新欄位才改：DDL cell 最後加欄位 + [c21] `ADD_COLUMNS` 登記 → 消費 job [c04] payload schema → [c06] transform。不要 DROP 重建。 |
| 爬蟲加 record_type | bronze 自動收（未登記的 record_type 一樣進 `b_*_record`）。要查才建 silver 表：加 DDL cell、[c04] `TABLE_KEYS`、[c06] transform 並登記到 `SILVER_TRANSFORMS`。 |
| 主鍵 | Delta 沒有 PK 約束；silver 的鍵寫在各 DDL cell 註解，消費 job [c04] `TABLE_KEYS` 用同一組鍵 MERGE。**改鍵要兩邊一起改。** |
| 不分割區 | 都是小表（bronze 每天數十列）。`b_record` 用 liquid clustering `CLUSTER BY (record_type)`；文件解析三張表 `CLUSTER BY (category, company_key)`（查詢都是「某分類 / 某公司」）；silver 開 autoOptimize。 |
| 時間欄位 | `TIMESTAMP` 一律存 UTC 瞬間（爬蟲的 `+08:00` 與無時區字串在 silver 轉換時統一處理）。`DATE`（含 `conference_date`）為**台北曆日**：爬蟲端已統一換算成台北時區，Databricks 不做任何時區處理。 |
| 直接查 bronze | Databricks SQL 可用路徑運算子：`SELECT payload:conference_date, payload:company_name FROM b_ir_calendar_record WHERE record_type = 'ir_conference'`。 |
| 對應文件 | `ir_calendar/doc/20260918_pass_data_to_databricks.md` 第 7 節（manifest 格式）、第 9 節（各類產出）；`doc/20260825_系統後端DB的Schema規劃.md`（系統端表，本 notebook 不建那些表）。 |


In [ ]:
# [c01] params
# 預設值取自 config/project.yml（catalog / schema）；job 執行時由 job parameters 覆蓋。
# 注意：widget 一旦在 notebook 建立過，改 code 的預設值不會更新既有 widget；要重設請先 dbutils.widgets.removeAll() 再跑本 cell。
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("domain", "ir_calendar")      # 表名中段；層級前綴 b_ / s_ 固定，見 docs/conventions.md 2.1
# 危險開關：只有填 YES_DROP_ALL 才會先 DROP 再建（資料全丟）。日常請留空。
dbutils.widgets.text("recreate", "")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
domain = dbutils.widgets.get("domain").strip()
recreate = dbutils.widgets.get("recreate").strip() == "YES_DROP_ALL"
assert catalog and schema and domain, "catalog / schema / domain 不可為空"
print(f"目標 schema = {catalog}.{schema}，domain = {domain!r}，recreate = {recreate}")


In [ ]:
# [c02] imports
# 表清單（順序即建立順序）：(層級, 短名)。完整表名 = <層級前綴><domain>_<短名>，如 b_ir_calendar_record、s_ir_calendar_conference。
LAYER_PREFIX = {"bronze": "b_", "silver": "s_", "gold": "g_"}
TABLES = [
    ("bronze", "record"),          # 爬蟲批次每個檔案一列，payload = JSON 全文；append-only
    ("bronze", "batch_log"),       # 每批處理紀錄（manifest 摘要 + 處理結果 + API 回應）
    ("silver", "conference"),      # calendar/*.json：法說場次，每個 revision 一列
    ("silver", "summary"),         # summaries/*.json：法說彙整
    ("silver", "document_file"),   # 實體檔案索引：落到 Volume 的每個 PDF / HTM
    ("silver", "company"),         # master/company.json：公司主表快照
    ("silver", "financials"),      # financials/*.json：財報數據（年度 + 季別，一期一列）；2026-10 新增
    # 以下三張由 ir_calendar_parse_documents 寫入（Volume 財報檔案的 ai_parse_document 結果）
    ("bronze", "document_parse"),  # 一檔一版一列，payload = ai_parse_document 輸出 JSON 全文
    ("silver", "document_text"),   # 一檔一列，全文 markdown
    ("silver", "document_element"),  # 一元素（段落 / 表格 / 圖表描述）一列
    # 以下兩張由 ir_calendar_build_ir_report 寫入（財報＋法說整合報告：supplier / peer / semi 三視角）；2026-10-20 新增，2026-10-29 自 supplier_report 改名（[c19c]）
    ("bronze", "ir_report_llm"),  # 每次 LLM 呼叫一列，payload = 原始回應；append-only
    ("gold", "ir_report"),        # 一家公司一期一列：HTML 全文 + 結構化 JSON；MERGE
]
DDLS: dict[tuple[str, str], str] = {}      # (層級, 短名) → CREATE TABLE 語句，由 [c10]～[c19b] 填入


def full_name(layer: str, short: str) -> str:
    return f"{catalog}.{schema}.{LAYER_PREFIX[layer]}{domain}_{short}"


# 小表共用屬性：自動合併小檔案；不分割區。
TBLPROPS = (
    "TBLPROPERTIES ("
    "'delta.autoOptimize.optimizeWrite' = 'true', "
    "'delta.autoOptimize.autoCompact' = 'true')"
)



In [ ]:
# [c10] ddl_b_record
# bronze 主表：爬蟲批次 manifest.files[] 的每個檔案一列（外加 manifest 本身一列，record_type = batch_manifest）。
# append-only；重跑同一批時先 DELETE WHERE batch_id 再 append（冪等）。schema 不隨爬蟲欄位變動：新欄位都在 payload 裡。
# 沒有主鍵；(batch_id, batch_path) 在正常情況下唯一。
DDLS[("bronze", "record")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("bronze", "record")} (
  batch_id     STRING    NOT NULL COMMENT '批次目錄名 <6碼序號>_<yyyyMMddTHHmmss>',
  seq          INT       NOT NULL COMMENT '批次序號（manifest.seq）；silver 取最新版本靠它排序',
  record_type  STRING    NOT NULL COMMENT 'batch_manifest / ir_conference / ir_summary / ir_document / ir_document_file / api_payload / app_company / app_company_category …',
  batch_path   STRING    NOT NULL COMMENT '批次內相對路徑，如 calendar/2026-09-18_KRX005930_2026Q3.json',
  company_key  STRING             COMMENT '<市場>:<代號>（manifest 條目有帶才填）',
  period       STRING             COMMENT '曆年期別 yyyyQn（manifest 條目有帶才填）',
  sha256       STRING             COMMENT '檔案內容雜湊（manifest.files[].sha256，下載時已核對）',
  bytes        BIGINT             COMMENT '檔案大小',
  volume_path  STRING             COMMENT '實體檔案（ir_document_file）落地的 Volume 完整路徑；JSON 類為 NULL',
  payload      STRING    NOT NULL COMMENT 'JSON 全文：JSON 檔是檔案內容；實體檔是 manifest 該條目。可用 payload:欄位 路徑查詢',
  ingested_at  TIMESTAMP NOT NULL COMMENT '寫入時間（UTC）'
)
USING DELTA
CLUSTER BY (record_type)
COMMENT 'ir_calendar bronze：爬蟲批次原樣落地，一檔一列，append-only'
"""


In [ ]:
# [c11] ddl_b_batch_log
# 主鍵：batch_id。job 遙測（不是來源資料），所以允許 MERGE 覆蓋：失敗先寫 FAILED，重跑成功後覆蓋成 SUCCESS。
# API 回應併在 api_results 陣列，不另開表；查失敗明細用 explode(api_results)。
# crawler_alerts 是 2026-10 加的欄位，放最後：既有表由 [c21] ALTER TABLE ADD COLUMNS 補，位置一致。
DDLS[("bronze", "batch_log")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("bronze", "batch_log")} (
  batch_id         STRING    NOT NULL COMMENT '批次目錄名，主鍵',
  seq              INT       NOT NULL COMMENT '批次序號（manifest.seq）',
  prev_seq         INT                COMMENT '前一批序號；第一批為 NULL',
  kind             STRING             COMMENT 'incremental（日常）/ master（主檔全量快照）',
  producer_mode    STRING             COMMENT '爬蟲執行模式：daily_scan / post_event / fetch_docs / master',
  producer_host    STRING             COMMENT '爬蟲主機名',
  producer_git_sha STRING             COMMENT '爬蟲程式版本',
  generated_at     TIMESTAMP          COMMENT '爬蟲端產出時間（UTC）',
  counts           MAP<STRING, INT>   COMMENT '各 record_type 筆數（manifest.counts）',
  files_total      INT                COMMENT 'manifest.files 總數',
  landed_files     INT                COMMENT '落到 Volume 的實體檔案數',
  archived_files   INT                COMMENT '歸檔到 <volume_root>/ir_calendar/batches/ 的 JSON 數',
  bronze_rows      INT                COMMENT '寫進 b_*_record 的列數',
  api_results      ARRAY<STRUCT<
                     endpoint: STRING, idempotency_key: STRING, rows: INT, http_status: INT, success: BOOLEAN,
                     created: INT, updated: INT, unchanged: INT, failed: INT,
                     failures_json: STRING, response_json: STRING, posted_at: TIMESTAMP>>
                                      COMMENT '每次 POST 系統 API 的結果；未轉送為空陣列',
  status           STRING    NOT NULL COMMENT 'SUCCESS / FAILED',
  error            STRING             COMMENT '失敗時的例外訊息（前 2000 字）',
  job_run_id       STRING             COMMENT 'Databricks job run id（由 job parameter 帶入）',
  processed_at     TIMESTAMP NOT NULL COMMENT '本 job 處理時間（UTC）',
  crawler_alerts   ARRAY<STRING>      COMMENT '本次 job 讀到的爬蟲端告警，[FATAL]/[WARN]/[INFO] 前綴；放棄明細一筆一行'
)
USING DELTA
COMMENT 'ir_calendar bronze：每個爬蟲批次一列的處理紀錄'
{TBLPROPS}
"""


In [ ]:
# [c12] ddl_s_conference
# 主鍵：(company_key, period, revision)。同一場次改期或補連結會出新 revision，保留歷史；
# 要「目前版本」取每個 (company_key, period) 的 max(revision)。系統端的鍵是 (stock_code, fiscal_period = period)。
# 來源：bronze record_type = ir_conference，由消費 job [c06] transform_conference 產生。
DDLS[("silver", "conference")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("silver", "conference")} (
  company_key          STRING    NOT NULL COMMENT '<市場>:<代號>，主鍵之一',
  company_name         STRING             COMMENT '公司名稱',
  english_name         STRING             COMMENT '英文名',
  stock_code           STRING             COMMENT '股票代號',
  market               STRING             COMMENT '交易所代碼',
  category             STRING             COMMENT '公司分類代碼',
  period               STRING    NOT NULL COMMENT '曆年期別 yyyyQn（如 2026Q2），主鍵之一；即系統端 fiscal_period',
  fiscal_period        STRING             COMMENT '爬蟲舊欄位：可能只有 Qn 不帶年，勿當鍵',
  source_fiscal_period STRING             COMMENT '來源公司自稱的期別（如 FY2026Q3），可為 NULL',
  conference_date      DATE               COMMENT '法說日期：台北曆日（爬蟲端已統一換算，本 job 不做任何時區處理）',
  start_time           STRING             COMMENT '開始時間（多為 NULL）',
  end_time             STRING             COMMENT '結束時間（實測 0% 可得）',
  conference_type      STRING             COMMENT 'ONSITE / ONLINE / PHONE / HYBRID / NULL',
  location             STRING             COMMENT '實體地點',
  meeting_link         STRING             COMMENT '線上會議連結',
  document_url         STRING             COMMENT '簡報 / 文件連結',
  status               STRING             COMMENT '爬蟲一律 SCHEDULED；COMPLETED 由系統端翻轉',
  importance           INT                COMMENT '重要性 1~5，由分類推導的初始值',
  source               STRING             COMMENT '來源：MOPS / TWSE / TPEX / LLM_SEARCH / SEED_ONETIME …',
  source_url           STRING             COMMENT '來源網址',
  confidence           STRING             COMMENT 'CONFIRMED（官方）/ TENTATIVE（轉述）',
  remark               STRING             COMMENT '備註',
  recipients           ARRAY<STRING>      COMMENT '通知對象代碼',
  revision             INT       NOT NULL COMMENT '事件版本號，主鍵之一',
  crawled_at           TIMESTAMP          COMMENT '爬取時間（UTC）',
  batch_id             STRING             COMMENT '來源批次（bronze 血緣）',
  seq                  INT                COMMENT '來源批次序號',
  batch_path           STRING             COMMENT '批次內路徑',
  updated_at           TIMESTAMP NOT NULL COMMENT 'silver 寫入 / 更新時間（UTC）'
)
USING DELTA
COMMENT 'ir_calendar silver：法說場次（對應系統端 app_ir_conference；每個 revision 一列）'
{TBLPROPS}
"""


In [ ]:
# [c13] ddl_s_summary
# 主鍵：(company_key, period)，同鍵以較新的批次為準。系統端沒有對應表，這裡是唯一的結構化存放處（規劃書第 9 節「不要丟」）。
# summary 物件欄位由 LLM 產出、可能增減：展開常用欄位，另存 summary_json 原文（也可回 bronze 取 payload）。
# headline_metrics 以後五欄是 2026-10 爬蟲端新增的選填欄位（MI_Crawler 交代文件 20261010 §2.1），放在 updated_at 之後：
# 既有表由 [c21] ALTER TABLE ADD COLUMNS 補，位置一致。舊批次沒有這些鍵 → NULL；rebuild_silver 可從 bronze 補算。
# reported_period / business_segments / qa_highlights 是 2026-10-21 爬蟲新版（對齊 SCM 報告新版面，交代文件 20261021 §1.2）再加的三欄，
# 同樣放最後；management_views.topic 的詞彙同時換成新版面六類（struct 不變，只是值）。
# value / yoy 是字串（保留原幣別與單位，如 NT$87.56 億、-3.78%），刻意不轉數值。
DDLS[("silver", "summary")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("silver", "summary")} (
  company_key     STRING    NOT NULL COMMENT '<市場>:<代號>，主鍵之一',
  company_name    STRING             COMMENT '公司名稱',
  stock_code      STRING             COMMENT '股票代號',
  market          STRING             COMMENT '交易所代碼',
  category        STRING             COMMENT '公司分類代碼',
  period          STRING    NOT NULL COMMENT '曆年期別 yyyyQn，主鍵之一（場次識別鍵；法說實際報告的期間看 reported_period）',
  fiscal_period   STRING             COMMENT '爬蟲舊欄位，與 period 同值，勿當鍵',
  conference_date DATE               COMMENT '法說日期：台北曆日（爬蟲端已統一換算，本 job 不做任何時區處理）',
  fallback        BOOLEAN            COMMENT 'true = 找不到法說內容、改以財報新聞稿彙整',
  found           BOOLEAN            COMMENT 'summary.found',
  core_points     ARRAY<STRING>      COMMENT '重點',
  guidance        STRING             COMMENT '展望 / 指引',
  key_numbers     ARRAY<STRING>      COMMENT '關鍵數字',
  risks           ARRAY<STRING>      COMMENT '風險',
  notes           STRING             COMMENT '彙整備註（期別對應、來源限制、媒體說法不一、是否提及 AUO 等）',
  sources         ARRAY<STRUCT<title: STRING, url: STRING, media: STRING>> COMMENT '引用來源',
  summary_json    STRING             COMMENT 'summary 物件原文（JSON 字串）',
  recipients      ARRAY<STRING>      COMMENT '通知對象代碼',
  crawled_at      TIMESTAMP          COMMENT '彙整時間（UTC）',
  batch_id        STRING             COMMENT '來源批次（bronze 血緣）',
  seq             INT                COMMENT '來源批次序號',
  batch_path      STRING             COMMENT '批次內路徑',
  updated_at      TIMESTAMP NOT NULL COMMENT 'silver 寫入 / 更新時間（UTC）',
  headline_metrics ARRAY<STRUCT<metric: STRING, period: STRING, value: STRING, yoy: STRING, note: STRING>>
                                     COMMENT '重點財務數字；metric 固定詞彙：營收/毛利率/營業利益率/稅後淨利/EPS/營業現金流/資本支出/負債比/其他',
  management_views ARRAY<STRUCT<topic: STRING, view: STRING>>
                                     COMMENT '經營層觀點；topic 固定詞彙（2026-10-21 起六類＝SCM 報告新版面法說會 01–06）：事業體／產品線重點、擴產／產能變化、獲利能力變化、未來市場預測、投資／M&A／組織整改、AUO 相關議題；10-21 前的舊批次為舊七類',
  positive_signals ARRAY<STRING>     COMMENT '正面訊號',
  watch_items      ARRAY<STRING>     COMMENT '觀察事項',
  open_questions   ARRAY<STRING>     COMMENT '待釐清問題',
  reported_period  STRING            COMMENT '這場法說實際報告的業績期間（公司自己的稱呼，如「2026 H1」「2026 Q3 及前三季自結」）；報告頁首用這欄，不用 period',
  business_segments ARRAY<STRUCT<segment: STRING, revenue: STRING, revenue_yoy: STRING, operating_margin: STRING, opm_yoy: STRING, note: STRING>>
                                     COMMENT '事業群／產品線表（新版面 03）；全部字串，台股簡報多半只有占比或營益率、revenue 常為 null',
  qa_highlights    ARRAY<STRUCT<question: STRING, answer: STRING>>
                                     COMMENT 'Q&A 重點（新版面 07）；只有電話會議型法說拿得到，台股券商場次多為空'
)
USING DELTA
COMMENT 'ir_calendar silver：法說彙整（LLM 產出；系統端無對應表）'
{TBLPROPS}
"""


In [ ]:
# [c14] ddl_s_document_file
# 主鍵：volume_path（Volume 上的完整路徑）。同名檔案 = 新 revision 覆蓋，這裡也覆蓋成最新的 sha256；舊版仍在 bronze。
# 公司 / 分類 / 期別 / 種類都取自 manifest 欄位，不拆檔名（規劃書 16.4）。
DDLS[("silver", "document_file")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("silver", "document_file")} (
  volume_path  STRING    NOT NULL COMMENT 'Volume 完整路徑 <volume_root>/<分類目錄>/<代稱>/<檔名>，主鍵',
  file_name    STRING             COMMENT '交付檔名 <代稱>_Q<季>CY<年>_<種類>.<副檔名>',
  company_key  STRING             COMMENT '<市場>:<代號>',
  company_slug STRING             COMMENT '公司代稱（Volume 目錄名）',
  category     STRING             COMMENT '公司分類代碼：PANEL_PEER / CUSTOMER / SUPPLIER',
  period       STRING             COMMENT '曆年期別 yyyyQn',
  doc_kind     STRING             COMMENT 'Presentation / Financial_Statements / Press_Release / Earnings_Transcript',
  fiscal_label STRING             COMMENT '公司口徑期別（如 FY2026Q3），可為 NULL',
  sha256       STRING             COMMENT '檔案內容雜湊（下載時已核對）',
  bytes        BIGINT             COMMENT '檔案大小',
  source_file  STRING             COMMENT '爬蟲工作面原檔名',
  source_url   STRING             COMMENT '原始下載網址',
  doc_date     DATE               COMMENT '文件日期，可為 NULL',
  batch_id     STRING             COMMENT '來源批次（bronze 血緣）',
  seq          INT                COMMENT '來源批次序號',
  batch_path   STRING             COMMENT '批次內路徑，如 docs/HK_1070/2026Q2/TCL-Electronics_Q2CY26_Presentation.pdf',
  updated_at   TIMESTAMP NOT NULL COMMENT 'silver 寫入 / 更新時間（UTC）'
)
USING DELTA
COMMENT 'ir_calendar silver：實體檔案索引，Volume 上每個 PDF / HTM 一列'
{TBLPROPS}
"""


In [ ]:
# [c15] ddl_s_company
# 主鍵：company_key（<市場>:<代號>，如 TPE:2353）。系統端用 stock_code 當鍵，但爬蟲的不變鍵是 company_key。
# 來源：bronze record_type = app_company（主檔批次的 master/company.json，rows[] 展開）；被移出名單的公司不會自動刪除，看 is_active。
# 公司分類常數（app_company_category）不另建 silver：只有 3 列，category_name 已在本表；要看原文查 bronze。
DDLS[("silver", "company")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("silver", "company")} (
  company_key         STRING    NOT NULL COMMENT '<市場>:<股票代號>，如 TPE:2353、NYSE:HPQ，主鍵',
  company_name        STRING             COMMENT '公司名稱（台股為中文法定名）',
  english_name        STRING             COMMENT '英文名（TWSE 來源可能被截為 40 字）',
  stock_code          STRING             COMMENT '股票代號；系統 API 的 upsert 鍵',
  market              STRING             COMMENT '交易所代碼：TPE / TYO / SZ / SH / KRX / HK / NYSE / NASDAQ / ETR',
  market_type         STRING             COMMENT 'LISTED / OTC / EMERGING；海外多為硬套或 NULL',
  category_name       STRING             COMMENT '公司分類代碼：PANEL_PEER / CUSTOMER / SUPPLIER',
  industry            STRING             COMMENT '產業別（台股官方代碼、海外人工分類，基準不一）',
  website_url         STRING             COMMENT '官網',
  ir_url              STRING             COMMENT '投資人關係頁',
  aliases             ARRAY<STRING>      COMMENT '別名清單（不唯一，不可當鍵）',
  recipients          ARRAY<STRING>      COMMENT '通知對象代碼（A00 / B00 …）',
  remark              STRING             COMMENT '備註',
  is_active           BOOLEAN            COMMENT '是否仍追蹤；已下市為 false',
  profile_source      STRING             COMMENT '主檔資料來源說明',
  file_slug           STRING             COMMENT '交付檔名用的公司代稱（data/company_slugs.json），可為 NULL',
  master_generated_at TIMESTAMP          COMMENT '主檔產出時間（UTC）',
  batch_id            STRING             COMMENT '來源批次（bronze 血緣）',
  seq                 INT                COMMENT '來源批次序號',
  batch_path          STRING             COMMENT '批次內路徑（master/company.json）',
  updated_at          TIMESTAMP NOT NULL COMMENT 'silver 寫入 / 更新時間（UTC）'
)
USING DELTA
COMMENT 'ir_calendar silver：公司主表快照（對應系統端 app_company；以 company_key 為鍵）'
{TBLPROPS}
"""


In [ ]:
# [c16] ddl_b_document_parse
# bronze：Volume 財報檔案經 ai_parse_document 的原始結果，一檔一版（volume_path, sha256）一列，payload = 輸出 JSON 全文。
# 由 ir_calendar_parse_documents 寫入：同鍵重解析時先 DELETE 再 append（冪等）；同路徑新版本檔（sha256 不同）是新的一列，舊版保留。
# category / company_key 是欄位不是表：新分類自動進來。CLUSTER BY 讓「某分類 / 某公司」的查詢只讀自己那塊。
DDLS[("bronze", "document_parse")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("bronze", "document_parse")} (
  volume_path    STRING    NOT NULL COMMENT 'Volume 完整路徑（= s_document_file.volume_path）',
  sha256         STRING             COMMENT '檔案內容雜湊；(volume_path, sha256) 識別一個檔案的一個版本',
  file_name      STRING             COMMENT '交付檔名',
  company_key    STRING             COMMENT '<市場>:<代號>',
  company_slug   STRING             COMMENT '公司代稱（Volume 目錄名）',
  category       STRING             COMMENT '公司分類代碼：PANEL_PEER / CUSTOMER / SUPPLIER…（新分類直接進來）',
  period         STRING             COMMENT '曆年期別 yyyyQn',
  doc_kind       STRING             COMMENT 'Presentation / Financial_Statements / Press_Release / Earnings_Transcript',
  fiscal_label   STRING             COMMENT '公司口徑期別，可為 NULL',
  bytes          BIGINT             COMMENT '檔案大小',
  parser         STRING             COMMENT '解析器：ai_parse_document',
  parser_version STRING             COMMENT 'ai_parse_document 的 version 選項',
  parse_options  STRING             COMMENT '呼叫時的完整選項 map（JSON，鍵排序）',
  attempt        INT       NOT NULL COMMENT '第幾次嘗試（含本次）；達到 max_attempts 後不再自動重試',
  status         STRING    NOT NULL COMMENT 'success / partial（部分頁失敗，error 存 error_status）/ failed（呼叫失敗、找不到檔、無結果）',
  error          STRING             COMMENT '失敗原因（截 2000 字）',
  page_count     INT                COMMENT 'payload.document.pages 數',
  element_count  INT                COMMENT 'payload.document.elements 數',
  text_chars     BIGINT             COMMENT '全文字元數（元素 content / description 串接後）',
  payload        STRING             COMMENT 'ai_parse_document 輸出 JSON 全文（CAST AS STRING）；failed 為 NULL。可用 payload:document.elements 路徑查',
  parsed_at      TIMESTAMP NOT NULL COMMENT '解析時間（UTC）',
  job_run_id     STRING             COMMENT 'Databricks job run id'
)
USING DELTA
CLUSTER BY (category, company_key)
COMMENT 'ir_calendar bronze：Volume 財報檔案的 ai_parse_document 原始結果，一檔一版一列；同鍵重解析時刪後 append'
"""


In [ ]:
# [c17] ddl_s_document_text
# 主鍵：(volume_path, sha256)。一檔一列，text_md 是元素依序串接的全文 markdown（圖表以描述代替）。
# 從 b_document_parse 純函式重算（parse_documents job 的 rebuild_silver）。
DDLS[("silver", "document_text")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("silver", "document_text")} (
  volume_path    STRING    NOT NULL COMMENT 'Volume 完整路徑，主鍵之一',
  sha256         STRING             COMMENT '檔案版本，主鍵之一',
  file_name      STRING             COMMENT '交付檔名',
  company_key    STRING             COMMENT '<市場>:<代號>',
  company_slug   STRING             COMMENT '公司代稱',
  category       STRING             COMMENT '公司分類代碼',
  period         STRING             COMMENT '曆年期別 yyyyQn',
  doc_kind       STRING             COMMENT '文件種類',
  fiscal_label   STRING             COMMENT '公司口徑期別',
  bytes          BIGINT             COMMENT '檔案大小',
  page_count     INT                COMMENT '頁數',
  element_count  INT                COMMENT '元素數',
  text_md        STRING             COMMENT '全文 markdown：元素依文件順序串接，圖表用描述代替',
  text_chars     BIGINT             COMMENT 'text_md 字元數',
  parser         STRING             COMMENT '解析器',
  parser_version STRING             COMMENT '解析器版本',
  parse_options  STRING             COMMENT '解析選項（JSON）',
  parse_status   STRING             COMMENT 'bronze.status：success / partial',
  parsed_at      TIMESTAMP          COMMENT 'bronze 解析時間（UTC）',
  updated_at     TIMESTAMP NOT NULL COMMENT 'silver 寫入 / 更新時間（UTC）'
)
USING DELTA
CLUSTER BY (category, company_key)
COMMENT 'ir_calendar silver：財報檔案全文（markdown），一檔一版一列'
{TBLPROPS}
"""


In [ ]:
# [c18] ddl_s_document_element
# 主鍵：(volume_path, sha256, element_id)。一元素一列：段落 / 標題 / 表格 / 圖表描述，給切塊、表格抽取用。
# 寫入是「以文件為單位刪後 append」而非逐列 MERGE：重新解析後元素數會變，MERGE 清不掉多出來的舊元素。
DDLS[("silver", "document_element")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("silver", "document_element")} (
  volume_path    STRING    NOT NULL COMMENT 'Volume 完整路徑，主鍵之一',
  sha256         STRING             COMMENT '檔案版本，主鍵之一',
  file_name      STRING             COMMENT '交付檔名',
  company_key    STRING             COMMENT '<市場>:<代號>',
  company_slug   STRING             COMMENT '公司代稱',
  category       STRING             COMMENT '公司分類代碼',
  period         STRING             COMMENT '曆年期別 yyyyQn',
  doc_kind       STRING             COMMENT '文件種類',
  fiscal_label   STRING             COMMENT '公司口徑期別',
  bytes          BIGINT             COMMENT '檔案大小',
  element_id     INT       NOT NULL COMMENT 'ai_parse_document 的元素 id（沒有就用 seq），主鍵之一',
  seq            INT                COMMENT '元素在文件內的順序（0 起）',
  page_id        INT                COMMENT '所在頁（0 起）',
  element_type   STRING             COMMENT 'text / section_header / table / figure / …（ai_parse_document 的 type）',
  content        STRING             COMMENT '元素內容（表格為 markdown / HTML）',
  description    STRING             COMMENT '圖表描述（只有開 descriptionElementTypes 的 doc_kind 才有）',
  parser         STRING             COMMENT '解析器',
  parser_version STRING             COMMENT '解析器版本',
  parse_options  STRING             COMMENT '解析選項（JSON）',
  parse_status   STRING             COMMENT 'bronze.status：success / partial',
  parsed_at      TIMESTAMP          COMMENT 'bronze 解析時間（UTC）',
  updated_at     TIMESTAMP NOT NULL COMMENT 'silver 寫入 / 更新時間（UTC）'
)
USING DELTA
CLUSTER BY (category, company_key)
COMMENT 'ir_calendar silver：財報檔案的解析元素，一元素一列'
{TBLPROPS}
"""


In [ ]:
# [c19] ddl_s_financials
# 主鍵：(company_key, period)，同鍵以較新的批次為準（爬蟲有新期別就整檔重發）。period：年度 2025FY、季別 2026Q2（曆年季）。
# 來源 financials/<市場_代號>.json（record_type = ir_financials）的 annual[] + quarterly[]，一期一列，period_type 區分。
# v1 只有台股（MOPS 三表，TWD 仟元）與美股（SEC companyfacts，USD 元）；跨市場比較要自己依 currency / unit 換算。
# 比率由爬蟲端從金額算好（%）；derived 標示推算值（台股 Q4 單季 = 全年 − Q3 累計；3M 毛利 = 營收 − 銷貨成本；季別單季現金流 = 本季累計 − 上季累計）。
# cash 以後 21 欄是 2026-10-21 爬蟲新版（對齊 SCM 報告新版面：速動比、應收／應付天數、母公司口徑淨利率）加的，放在 updated_at 之後：
# 既有表由 [c21] ALTER TABLE ADD COLUMNS 補，位置一致；新版上線前的批次這些欄位為 NULL，爬蟲整檔重發後 rebuild_silver 即有值。
DDLS[("silver", "financials")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("silver", "financials")} (
  company_key             STRING    NOT NULL COMMENT '<市場>:<代號>，主鍵之一',
  period                  STRING    NOT NULL COMMENT '年度 yyyyFY / 季別 yyyyQn（曆年季），主鍵之一',
  period_type             STRING    NOT NULL COMMENT 'annual / quarterly',
  fiscal_year             INT                COMMENT '年度列的會計年度；季別列為 NULL',
  period_end              DATE               COMMENT '期末日',
  company_name            STRING             COMMENT '公司名稱',
  stock_code              STRING             COMMENT '股票代號',
  market                  STRING             COMMENT '交易所代碼',
  category                STRING             COMMENT '公司分類代碼',
  currency                STRING             COMMENT '幣別：TWD / USD',
  unit                    STRING             COMMENT '金額單位：thousand（台股仟元）/ one（美股元）；EPS 一律為元',
  source                  STRING             COMMENT '資料來源：MOPS t164sb03/04/05 / SEC companyfacts',
  revenue                 DOUBLE             COMMENT '營收（單季 / 全年）',
  cost_of_revenue         DOUBLE             COMMENT '銷貨成本（美股原有；台股 2026-10-21 起「營業成本合計」）',
  gross_profit            DOUBLE             COMMENT '毛利',
  operating_income        DOUBLE             COMMENT '營業利益',
  pretax_income           DOUBLE             COMMENT '稅前淨利',
  net_income              DOUBLE             COMMENT '本期淨利（合併，含非控制權益）',
  net_income_parent       DOUBLE             COMMENT '歸屬母公司淨利（台股年度列 2026-10-21 起才有值；美股與 net_income 同值）',
  eps                     DOUBLE             COMMENT 'EPS（元）；推算的 Q4 為 NULL',
  total_assets            DOUBLE             COMMENT '資產總額（期末）',
  total_liabilities       DOUBLE             COMMENT '負債總額（期末）',
  current_assets          DOUBLE             COMMENT '流動資產（期末）',
  current_liabilities     DOUBLE             COMMENT '流動負債（期末）',
  equity                  DOUBLE             COMMENT '權益總額（期末，含非控制權益）',
  operating_cash_flow     DOUBLE             COMMENT '營業現金流（年度列；季別列為單季，台股由累計推算、列在 derived）',
  investing_cash_flow     DOUBLE             COMMENT '投資現金流（年度列；季別列為單季推算）',
  capex                   DOUBLE             COMMENT '資本支出（照現金流量表原值，為負數；季別列為單季推算；美股 2025 起 companyfacts 無 tag → NULL）',
  revenue_ytd             DOUBLE             COMMENT '營收累計（季別列）',
  net_income_ytd          DOUBLE             COMMENT '淨利累計（季別列）',
  operating_cash_flow_ytd DOUBLE             COMMENT '營業現金流累計（季別列；MOPS 只有累計）',
  capex_ytd               DOUBLE             COMMENT '資本支出累計（季別列）',
  gross_margin            DOUBLE             COMMENT '毛利率 %',
  operating_margin        DOUBLE             COMMENT '營業利益率 %',
  net_margin              DOUBLE             COMMENT '淨利率 %（合併口徑）',
  debt_ratio              DOUBLE             COMMENT '負債比 %',
  current_ratio           DOUBLE             COMMENT '流動比（倍）',
  derived                 ARRAY<STRING>      COMMENT '推算值欄位名（非官方直接數字）',
  source_urls             ARRAY<STRING>      COMMENT '原始資料網址',
  generated_at            TIMESTAMP          COMMENT '爬蟲端產出時間（UTC）',
  batch_id                STRING             COMMENT '來源批次（bronze 血緣）',
  seq                     INT                COMMENT '來源批次序號',
  batch_path              STRING             COMMENT '批次內路徑',
  updated_at              TIMESTAMP NOT NULL COMMENT 'silver 寫入 / 更新時間（UTC）',
  cash                    DOUBLE             COMMENT '現金及約當現金（期末）',
  fin_assets_fvtpl_current     DOUBLE        COMMENT '透過損益按公允價值衡量之金融資產－流動（台股）',
  fin_assets_fvoci_current     DOUBLE        COMMENT '透過其他綜合損益按公允價值衡量之金融資產－流動（台股）',
  fin_assets_amortized_current DOUBLE        COMMENT '按攤銷後成本衡量之金融資產－流動（台股）',
  short_term_investments  DOUBLE             COMMENT '短期投資（美股 ShortTermInvestments 等；台股無此鍵）',
  notes_receivable        DOUBLE             COMMENT '應收票據淨額（台股）',
  accounts_receivable     DOUBLE             COMMENT '應收帳款淨額',
  accounts_receivable_related DOUBLE         COMMENT '應收帳款－關係人淨額（台股）',
  inventory               DOUBLE             COMMENT '存貨',
  notes_payable           DOUBLE             COMMENT '應付票據（台股）',
  accounts_payable        DOUBLE             COMMENT '應付帳款',
  accounts_payable_related DOUBLE            COMMENT '應付帳款－關係人（台股）',
  equity_parent           DOUBLE             COMMENT '歸屬於母公司業主之權益合計',
  investing_cash_flow_ytd DOUBLE             COMMENT '投資現金流累計（季別列）',
  pretax_margin           DOUBLE             COMMENT '稅前淨利率 %（稅前淨利 ÷ 營收）',
  net_margin_parent       DOUBLE             COMMENT '母公司淨利率 %（歸屬母公司淨利 ÷ 營收）；SCM 報告「稅後淨利率」用這欄',
  quick_assets            DOUBLE             COMMENT '速動資產 = 現金 + 流動金融資產 + 應收票據與帳款（含關係人）；只有現金或只有應收時為 NULL',
  quick_ratio             DOUBLE             COMMENT '速動比（倍）= 速動資產 ÷ 流動負債',
  dso_days                DOUBLE             COMMENT '應收天數（年度列）= 平均應收 ÷ 年度營收 × 365',
  dpo_days                DOUBLE             COMMENT '應付天數（年度列）= 平均應付 ÷ 年度銷貨成本 × 365',
  days_basis              STRING             COMMENT '天數的平均基準：avg = (本年底 + 上年底)/2；end = 上年度缺、用本年底'
)
USING DELTA
COMMENT 'ir_calendar silver：財報數據（台股 MOPS / 美股 SEC；年度 + 季別一期一列）'
{TBLPROPS}
"""


In [ ]:
# [c19a] ddl_b_ir_report_llm
# bronze：ir_calendar_build_ir_report 每次 LLM 呼叫一列（含失敗與 JSON 解析失敗），payload = 回應原文。append-only，不刪不 MERGE。
# 要看某份報告用了什麼輸出：WHERE company_key = … AND period = … ORDER BY called_at DESC。
DDLS[("bronze", "ir_report_llm")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("bronze", "ir_report_llm")} (
  company_key       STRING    NOT NULL COMMENT '<市場>:<代號>',
  period            STRING    NOT NULL COMMENT '曆年期別 yyyyQn',
  prompt_version    STRING             COMMENT '[c05] PROMPT_VERSION_IN_CELL',
  model             STRING             COMMENT 'LLM 模型名',
  attempt           INT                COMMENT '同一份報告的第幾次呼叫（1 = 首次、2 = JSON 重試）',
  docs_digest       STRING             COMMENT '輸入文件 sha256 清單的雜湊（對 g_ir_report.docs_digest）',
  status            STRING             COMMENT 'success / json_error / failed（端點失敗）',
  error             STRING             COMMENT '錯誤原因（截 2000 字）',
  stop_reason       STRING             COMMENT 'LLM stop_reason；max_tokens = 被截斷',
  prompt_chars      INT                COMMENT 'user prompt 字元數',
  input_tokens      INT                COMMENT '輸入 tokens',
  output_tokens     INT                COMMENT '輸出 tokens',
  cache_read_tokens INT                COMMENT 'prompt cache 命中 tokens',
  latency_s         DOUBLE             COMMENT '呼叫耗時（秒）',
  payload           STRING             COMMENT 'LLM 回應原文；端點失敗為 NULL',
  called_at         TIMESTAMP NOT NULL COMMENT '呼叫時間（UTC）',
  job_run_id        STRING             COMMENT 'Databricks job run id',
  report_kind       STRING             COMMENT '報告視角 supplier / peer / semi（2026-10-29 新增，既有表由 [c21] 補）'
)
USING DELTA
COMMENT 'ir_calendar bronze：供應商整合報告的 LLM 呼叫紀錄，一次呼叫一列；append-only'
{TBLPROPS}
"""



In [ ]:
# [c19b] ddl_g_ir_report
# gold：「財報＋法說整合報告」，一家公司一期一視角一列。鍵 (company_key, period, report_kind)，MERGE 覆蓋成最新版。
# 2026-10-29 視角：report_kind = supplier（採購）/ peer（面板同業）/ semi（半導體）；一家公司可多視角（build_ir_report [c02] company_views）。
# 只從 silver 建（docs/conventions.md 2.1）：數字來自 s_financials、文字由 LLM 讀 s_document_text 產出。
# html 全文放表裡（同 g_sales_brief_mail_report 作法，單份 < 1 MB），Volume 另存一份給人開。
DDLS[("gold", "ir_report")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("gold", "ir_report")} (
  company_key       STRING    NOT NULL COMMENT '<市場>:<代號>，主鍵之一',
  period            STRING    NOT NULL COMMENT '曆年期別 yyyyQn，主鍵之一（第三個鍵 report_kind 在最後，2026-10-29 加）',
  company_name      STRING             COMMENT '公司名稱',
  english_name      STRING             COMMENT '英文名',
  stock_code        STRING             COMMENT '股票代號',
  market            STRING             COMMENT '交易所代碼',
  category          STRING             COMMENT '公司分類代碼（SUPPLIER…）',
  company_slug      STRING             COMMENT '公司代稱（Volume 目錄名）',
  conference_date   STRING             COMMENT '法說日期（台北曆日，字串）；無場次為 NULL',
  report_title      STRING             COMMENT '報告標題',
  template_version  STRING             COMMENT '版型版本（[c02] TEMPLATE_VERSION）',
  prompt_version    STRING             COMMENT 'prompt 版本',
  model             STRING             COMMENT 'LLM 模型名',
  docs_digest       STRING             COMMENT '來源雜湊：帶入文件 sha256 + 彙整批次 + 場次 revision（2026-10-28 起）；任一變了 → 自動重做',
  doc_count         INT                COMMENT '輸入文件數',
  doc_kinds         ARRAY<STRING>      COMMENT '輸入文件種類',
  has_financials    STRING             COMMENT 'true / false：是否有結構化財報數據（無 → 表格圖表皆未取得）',
  annual_periods    INT                COMMENT '年度數列筆數（≤ 5）',
  quarterly_periods INT                COMMENT '季別數列筆數（≤ 5）',
  light             STRING             COMMENT '採購燈號 green / amber / red（LLM 示範分數）',
  score             INT                COMMENT '0～100 採購管理用示範分數，非信用評等',
  llm_json          STRING             COMMENT 'LLM 結構化輸出（normalize 後 JSON）；render_only 重套版用',
  html              STRING             COMMENT '完整報告 HTML（可單獨開啟）',
  html_path         STRING             COMMENT 'Volume 上的 HTML 路徑',
  html_chars        BIGINT             COMMENT 'HTML 字元數',
  input_tokens      INT                COMMENT '本份報告 LLM 輸入 tokens 合計',
  output_tokens     INT                COMMENT '本份報告 LLM 輸出 tokens 合計',
  generated_at      TIMESTAMP NOT NULL COMMENT '產出時間（UTC）',
  job_run_id        STRING             COMMENT 'Databricks job run id',
  source_periods    ARRAY<STRING>      COMMENT '帶入文件來自哪幾期（含前期補位；2026-10-28 新增，既有表由 [c21] 補）',
  summary_batch_id  STRING             COMMENT '採用的爬蟲法說彙整批次（s_summary.batch_id）；無彙整為 NULL',
  docs_fed          ARRAY<STRUCT<file_name: STRING, period: STRING, doc_kind: STRING, fed: STRING, chars_fed: INT>> COMMENT '每份文件餵 LLM 的狀態：full / truncated / skipped 與字元數',
  report_kind       STRING             COMMENT '報告視角 supplier / peer / semi，主鍵之一（2026-10-29 新增，既有表由 [c21] 補並回填 supplier）',
  topics            ARRAY<STRING>      COMMENT 'semi 視角的題目標籤：A 封裝／載板、B 光通訊／矽光子、C 景氣風向；其他視角為空'
)
USING DELTA
COMMENT 'ir_calendar gold：財報＋法說整合報告（supplier 採購 / peer 面板同業 / semi 半導體視角），一家公司一期一視角一列；MERGE'
{TBLPROPS}
"""



In [ ]:
# [c19c] rename_tables
# 2026-10-29 一次性改名：supplier_report → ir_report（報告不只供應商，含 peer / semi 視角；表結構不變，資料保留）。
# 舊名存在且新名不存在才 RENAME；都不存在由 [c20] 用新名建；重跑無害。**必須在 [c20] 之前**，否則 [c20] 會先建出空的新表、RENAME 就失敗。
RENAMES = [(("bronze", "supplier_report_llm"), ("bronze", "ir_report_llm")),
           (("gold", "supplier_report"), ("gold", "ir_report"))]
for (old_layer, old_short), (new_layer, new_short) in RENAMES:
    old, new = full_name(old_layer, old_short), full_name(new_layer, new_short)
    old_exists, new_exists = spark.catalog.tableExists(old), spark.catalog.tableExists(new)
    if old_exists and not new_exists:
        spark.sql(f"ALTER TABLE {old} RENAME TO {new}")
        print(f"renamed {old} → {new}")
    elif old_exists and new_exists:
        print(f"!!! 新舊都在：{old} 與 {new}。確認舊表 0 列後手動 DROP 舊表；[c20] / [c21] 只會動新表")
    else:
        print(f"ok  {new}（無需改名）")


In [ ]:
# [c20] run_create
# 建表順序照 TABLES。recreate 只在明確填 YES_DROP_ALL 時才 DROP，且先印出要刪的表。
missing = [t for t in TABLES if t not in DDLS]
assert not missing, f"缺 DDL：{missing}（對應 cell 沒跑到）"

if recreate:
    print("!!! recreate = YES_DROP_ALL，將刪除以下表與其資料：")
    for layer, short in TABLES:
        print("   ", full_name(layer, short))
    for layer, short in TABLES:
        spark.sql(f"DROP TABLE IF EXISTS {full_name(layer, short)}")

for layer, short in TABLES:
    spark.sql(DDLS[(layer, short)])
    print(f"ok  {full_name(layer, short)}")


In [ ]:
# [c21] add_columns
# 既有表補欄位：CREATE TABLE IF NOT EXISTS 不會改已存在的表，上線後才加的欄位在這裡 ALTER TABLE ADD COLUMNS。
# 欄位已存在就略過，可重跑。新建的表由 [c10]～[c19] 的 DDL 直接含這些欄位（同樣放在最後），兩邊欄位順序一致。
# 要先跑這格，再上線讀寫新欄位的 notebook（consume_batches 的 batch_log MERGE 不開 schema evolution，缺欄位會失敗）。
# 型別 / 註解與對應 DDL cell 一致：summary 看 [c13]、financials 看 [c19]。
_FIN_NEW = [  # 2026-10-21 爬蟲新版（交代文件 20261021 §1.1 / §3.1）
    ("cash", "DOUBLE", "現金及約當現金（期末）"),
    ("fin_assets_fvtpl_current", "DOUBLE", "透過損益按公允價值衡量之金融資產－流動（台股）"),
    ("fin_assets_fvoci_current", "DOUBLE", "透過其他綜合損益按公允價值衡量之金融資產－流動（台股）"),
    ("fin_assets_amortized_current", "DOUBLE", "按攤銷後成本衡量之金融資產－流動（台股）"),
    ("short_term_investments", "DOUBLE", "短期投資（美股 ShortTermInvestments 等；台股無此鍵）"),
    ("notes_receivable", "DOUBLE", "應收票據淨額（台股）"),
    ("accounts_receivable", "DOUBLE", "應收帳款淨額"),
    ("accounts_receivable_related", "DOUBLE", "應收帳款－關係人淨額（台股）"),
    ("inventory", "DOUBLE", "存貨"),
    ("notes_payable", "DOUBLE", "應付票據（台股）"),
    ("accounts_payable", "DOUBLE", "應付帳款"),
    ("accounts_payable_related", "DOUBLE", "應付帳款－關係人（台股）"),
    ("equity_parent", "DOUBLE", "歸屬於母公司業主之權益合計"),
    ("investing_cash_flow_ytd", "DOUBLE", "投資現金流累計（季別列）"),
    ("pretax_margin", "DOUBLE", "稅前淨利率 %（稅前淨利 ÷ 營收）"),
    ("net_margin_parent", "DOUBLE", "母公司淨利率 %（歸屬母公司淨利 ÷ 營收）；SCM 報告「稅後淨利率」用這欄"),
    ("quick_assets", "DOUBLE", "速動資產 = 現金 + 流動金融資產 + 應收票據與帳款（含關係人）；只有現金或只有應收時為 NULL"),
    ("quick_ratio", "DOUBLE", "速動比（倍）= 速動資產 ÷ 流動負債"),
    ("dso_days", "DOUBLE", "應收天數（年度列）= 平均應收 ÷ 年度營收 × 365"),
    ("dpo_days", "DOUBLE", "應付天數（年度列）= 平均應付 ÷ 年度銷貨成本 × 365"),
    ("days_basis", "STRING", "天數的平均基準：avg = (本年底 + 上年底)/2；end = 上年度缺、用本年底"),
]
ADD_COLUMNS: dict[tuple[str, str], list[tuple[str, str, str]]] = {
    # 2026-10-11 爬蟲端告警分級（MI_Crawler 交代文件 20261010 §3.1）
    ("bronze", "batch_log"): [
        ("crawler_alerts", "ARRAY<STRING>",
         "本次 job 讀到的爬蟲端告警，[FATAL]/[WARN]/[INFO] 前綴；放棄明細一筆一行"),
    ],
    # 2026-10-11 法說彙整五個新選填欄位（同交代文件 §2.1）＋ 2026-10-21 新版三欄（交代文件 20261021 §1.2）；型別 / 註解與 [c13] DDL 一致
    ("silver", "summary"): [
        ("headline_metrics",
         "ARRAY<STRUCT<metric: STRING, period: STRING, value: STRING, yoy: STRING, note: STRING>>",
         "重點財務數字；metric 固定詞彙：營收/毛利率/營業利益率/稅後淨利/EPS/營業現金流/資本支出/負債比/其他"),
        ("management_views", "ARRAY<STRUCT<topic: STRING, view: STRING>>",
         "經營層觀點；topic 固定詞彙（2026-10-21 起六類＝SCM 報告新版面法說會 01–06）：事業體／產品線重點、擴產／產能變化、獲利能力變化、未來市場預測、投資／M&A／組織整改、AUO 相關議題；10-21 前的舊批次為舊七類"),
        ("positive_signals", "ARRAY<STRING>", "正面訊號"),
        ("watch_items", "ARRAY<STRING>", "觀察事項"),
        ("open_questions", "ARRAY<STRING>", "待釐清問題"),
        ("reported_period", "STRING",
         "這場法說實際報告的業績期間（公司自己的稱呼，如「2026 H1」「2026 Q3 及前三季自結」）；報告頁首用這欄，不用 period"),
        ("business_segments",
         "ARRAY<STRUCT<segment: STRING, revenue: STRING, revenue_yoy: STRING, operating_margin: STRING, opm_yoy: STRING, note: STRING>>",
         "事業群／產品線表（新版面 03）；全部字串，台股簡報多半只有占比或營益率、revenue 常為 null"),
        ("qa_highlights", "ARRAY<STRUCT<question: STRING, answer: STRING>>",
         "Q&A 重點（新版面 07）；只有電話會議型法說拿得到，台股券商場次多為空"),
    ],
    ("silver", "financials"): _FIN_NEW,
    # 2026-10-28 build_ir_report 來源完整性（backlog D2）：與 [c19b] DDL 最後三欄一致
    ("gold", "ir_report"): [
        ("source_periods", "ARRAY<STRING>", "帶入文件來自哪幾期（含前期補位；2026-10-28 新增，既有表由 [c21] 補）"),
        ("summary_batch_id", "STRING", "採用的爬蟲法說彙整批次（s_summary.batch_id）；無彙整為 NULL"),
        ("docs_fed", "ARRAY<STRUCT<file_name: STRING, period: STRING, doc_kind: STRING, fed: STRING, chars_fed: INT>>",
         "每份文件餵 LLM 的狀態：full / truncated / skipped 與字元數"),
        # 2026-10-29 報告視角（與 [c19b] DDL 最後兩欄一致）
        ("report_kind", "STRING", "報告視角 supplier / peer / semi，主鍵之一（2026-10-29 新增，既有表由 [c21] 補並回填 supplier）"),
        ("topics", "ARRAY<STRING>", "semi 視角的題目標籤：A 封裝／載板、B 光通訊／矽光子、C 景氣風向；其他視角為空"),
    ],
    ("bronze", "ir_report_llm"): [
        ("report_kind", "STRING", "報告視角 supplier / peer / semi（2026-10-29 新增，既有表由 [c21] 補）"),
    ],
}
# 既有欄位只改註解（詞彙 / 口徑變了）：ALTER COLUMN COMMENT 可重跑
COLUMN_COMMENTS: dict[tuple[str, str], list[tuple[str, str]]] = {
    ("silver", "financials"): [
        ("cost_of_revenue", "銷貨成本（美股原有；台股 2026-10-21 起「營業成本合計」）"),
        ("net_income_parent", "歸屬母公司淨利（台股年度列 2026-10-21 起才有值；美股與 net_income 同值）"),
        ("operating_cash_flow", "營業現金流（年度列；季別列為單季，台股由累計推算、列在 derived）"),
        ("investing_cash_flow", "投資現金流（年度列；季別列為單季推算）"),
        ("capex", "資本支出（照現金流量表原值，為負數；季別列為單季推算；美股 2025 起 companyfacts 無 tag → NULL）"),
    ],
    ("gold", "ir_report"): [
        ("docs_digest", "來源雜湊：帶入文件 sha256 + 彙整批次 + 場次 revision（2026-10-28 起）；任一變了 → 自動重做"),
    ],
}

for (layer, short), cols in ADD_COLUMNS.items():
    t = full_name(layer, short)
    have = set(spark.table(t).columns)
    todo = [c for c in cols if c[0] not in have]
    if not todo:
        print(f"ok  {t}：欄位已齊")
        continue
    spec = ", ".join(f"{n} {typ} COMMENT '{cm}'" for n, typ, cm in todo)
    spark.sql(f"ALTER TABLE {t} ADD COLUMNS ({spec})")
    print(f"add {t}：{[c[0] for c in todo]}")

# 2026-10-29：gold 鍵多 report_kind，舊列（10-29 前產的供應商報告）回填 supplier，MERGE 才對得到鍵（<=> 對 NULL 不會匹配 'supplier'）
_g = full_name("gold", "ir_report")
n_null = spark.sql(f"SELECT count(*) AS n FROM {_g} WHERE report_kind IS NULL").collect()[0]["n"]
if n_null:
    spark.sql(f"UPDATE {_g} SET report_kind = 'supplier' WHERE report_kind IS NULL")
    print(f"backfill {_g}：report_kind = supplier × {n_null}")

for (layer, short), cols in COLUMN_COMMENTS.items():
    t = full_name(layer, short)
    for n, cm in cols:
        spark.sql(f"ALTER TABLE {t} ALTER COLUMN {n} COMMENT '{cm}'")
    print(f"comment {t}：{[c[0] for c in cols]}")



In [ ]:
# [c30] check_tables
# 驗證：每張表都存在、欄位數、目前列數（都是小表，count 可接受）。
rows = []
for layer, short in TABLES:
    t = full_name(layer, short)
    df_t = spark.table(t)
    rows.append((t, len(df_t.columns), df_t.count()))
w = max(len(r[0]) for r in rows)
for t, ncol, nrow in rows:
    print(f"{t:<{w}}  cols={ncol:>2}  rows={nrow}")
